In [ ]:
# Setup the Jupyter version of Dash
from collections import Counter
from pathlib import Path
import base64
import os

import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash
import pandas as pd
import plotly.express as px

from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()


###########################
# Database Configuration
###########################
username = os.getenv("MONGO_USERNAME", "aacuser")
password = os.getenv("MONGO_PASSWORD", "SNHU123456")
db = AnimalShelter(username=username, password=password)
db.create_indexes()

DISPLAY_FIELDS = [
    "animal_id",
    "name",
    "animal_type",
    "breed",
    "sex_upon_outcome",
    "age_upon_outcome_in_weeks",
    "outcome_type",
    "location_lat",
    "location_long",
]
DISPLAY_PROJECTION = {field: 1 for field in DISPLAY_FIELDS}
DISPLAY_PROJECTION["_id"] = 0


RESCUE_PROFILES = {
    "water": {
        "label": "Water Rescue",
        "breeds": frozenset(
            [
                "Labrador Retriever Mix",
                "Labrador Retriever",
                "Chesapeake Bay Retriever",
                "Newfoundland",
            ]
        ),
        "sex": "Intact Male",
        "age_range": (26, 156),
    },
    "mountain": {
        "label": "Mountain / Wilderness Rescue",
        "breeds": frozenset(
            [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler",
            ]
        ),
        "sex": "Intact Male",
        "age_range": (26, 156),
    },
    "disaster": {
        "label": "Disaster / Individual Tracking",
        "breeds": frozenset(
            [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler",
            ]
        ),
        "sex": "Intact Male",
        "age_range": (20, 300),
    },
}


def build_query(filter_type):
    profile = RESCUE_PROFILES.get(filter_type)
    if profile is None:
        return {}

    min_age, max_age = profile["age_range"]
    return {
        "animal_type": "Dog",
        "breed": {"$in": sorted(profile["breeds"])},
        "sex_upon_outcome": profile["sex"],
        "age_upon_outcome_in_weeks": {"$gte": min_age, "$lte": max_age},
    }


def fetch_records(filter_type):
    """Read dashboard records through a projection-limited database query."""
    return db.read(
        build_query(filter_type),
        projection=DISPLAY_PROJECTION,
        limit=500,
        sort=[("breed", 1), ("name", 1)],
    )


def records_to_frame(records):
    frame = pd.DataFrame.from_records(records)
    for field in DISPLAY_FIELDS:
        if field not in frame.columns:
            frame[field] = None
    return frame[DISPLAY_FIELDS]


def top_breed_counts(records, limit=10):
    counts = Counter((record.get("breed") or "Unknown") for record in records or [])
    return pd.DataFrame(counts.most_common(limit), columns=["breed", "count"])


def selected_index(selected_rows, row_count):
    if row_count <= 0:
        return None
    if not selected_rows:
        return 0
    try:
        candidate = int(selected_rows[0])
    except (TypeError, ValueError):
        return 0
    return candidate if 0 <= candidate < row_count else 0


def safe_float(value, fallback):
    try:
        return float(value)
    except (TypeError, ValueError):
        return fallback


df = records_to_frame(fetch_records("reset"))


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)


def load_logo():
    logo_path = Path("Grazioso Salvare Logo.png")
    if not logo_path.exists():
        return None
    with logo_path.open("rb") as logo_file:
        return base64.b64encode(logo_file.read()).decode()


logo_image = load_logo()
branding = []
if logo_image:
    branding.append(
        html.Img(
            src="data:image/png;base64,{}".format(logo_image),
            style={"height": "100px"},
        )
    )
branding.append(html.Div("John Rosario", style={"fontSize": "16px", "marginTop": "5px"}))

filter_options = [{"label": "Reset (All)", "value": "reset"}]
filter_options.extend(
    {"label": profile["label"], "value": key}
    for key, profile in sorted(RESCUE_PROFILES.items())
)

app.layout = html.Div(
    [
        html.Div(branding, style={"textAlign": "center"}),
        html.Center(html.B(html.H1("CS-340 Dashboard"))),
        html.Hr(),
        html.Div(
            [
                html.Label("Rescue Type Filter", style={"fontWeight": "bold"}),
                dcc.RadioItems(
                    id="filter-type",
                    options=filter_options,
                    value="reset",
                    inline=True,
                ),
            ],
            style={"margin": "10px 0"},
        ),
        html.Hr(),
        dash_table.DataTable(
            id="datatable-id",
            columns=[
                {"name": column, "id": column, "deletable": False, "selectable": True}
                for column in df.columns
            ],
            data=df.to_dict("records"),
            row_selectable="single",
            selected_rows=[],
            selected_columns=[],
            page_size=10,
            sort_action="native",
            filter_action="native",
            style_table={"overflowX": "auto"},
        ),
        html.Br(),
        html.Hr(),
        html.Div(
            className="row",
            style={"display": "flex"},
            children=[
                html.Div(id="graph-id", className="col s12 m6"),
                html.Div(id="map-id", className="col s12 m6"),
            ],
        ),
    ]
)


#############################################
# Interaction Between Components / Controller
#############################################
@app.callback(Output("datatable-id", "data"), [Input("filter-type", "value")])
def update_dashboard(filter_type):
    filtered_frame = records_to_frame(fetch_records(filter_type))
    return filtered_frame.to_dict("records")


@app.callback(Output("graph-id", "children"), [Input("datatable-id", "derived_virtual_data")])
def update_graphs(view_data):
    breed_counts = top_breed_counts(view_data)
    if breed_counts.empty:
        return []

    fig = px.pie(
        breed_counts,
        names="breed",
        values="count",
        title="Top 10 Breeds (Current Table View)",
    )
    return [dcc.Graph(figure=fig)]


@app.callback(Output("datatable-id", "style_data_conditional"), [Input("datatable-id", "selected_columns")])
def update_styles(selected_columns):
    return [
        {"if": {"column_id": column}, "background_color": "#D2F3FF"}
        for column in selected_columns
    ]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows"),
    ],
)
def update_map(view_data, selected_rows):
    frame = records_to_frame(view_data or [])
    row_index = selected_index(selected_rows, len(frame))
    if row_index is None:
        return []

    row = frame.iloc[row_index]
    breed = str(row.get("breed", "Unknown"))
    name = str(row.get("name", "Unknown"))

    default_lat, default_lon = 30.75, -97.48
    lat = safe_float(row.get("location_lat"), default_lat)
    lon = safe_float(row.get("location_long"), default_lon)

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([html.H1("Animal Name"), html.P(name)]),
                    ],
                ),
            ],
        )
    ]


In [ ]:
app.run_server(debug=False)
